# Observability, Monitoring & Alerting

A comprehensive guide to understanding system health, detecting incidents early, and building reliable alerting pipelines.

---

## Table of Contents
1. [Observability vs Monitoring — What's the Difference?](#1)
2. [The Three Pillars of Observability](#2)
3. [Metrics Deep Dive](#3)
4. [Logs Deep Dive](#4)
5. [Distributed Tracing Deep Dive](#5)
6. [What to Measure — Golden Signals, USE & RED Methods](#6)
7. [SLIs, SLOs, SLAs — The Reliability Contract](#7)
8. [Alerting — Turning Signals into Action](#8)
9. [Dashboards & Visualization](#9)
10. [Incident Detection Workflow](#10)
11. [Tooling Landscape](#11)
12. [Common Pitfalls](#12)

---


## 1. Observability vs Monitoring — What's the Difference?

<a name="1"></a>

These two terms are often used interchangeably but they describe fundamentally different concerns.

---

### Monitoring

**Monitoring** answers the question: *"Is this known thing broken?"*

You define in advance what you care about (CPU usage, error rate, queue depth), collect those numbers, and alert when a threshold is breached. Monitoring is **pre-defined and reactive** — it tells you *that* something is wrong.

```
Monitoring = Dashboards + Alerts on known failure modes
```

**Limitations of monitoring alone:**
- Can only catch failures you anticipated
- "Unknown unknowns" — novel failure modes go undetected
- Alert fatigue from threshold tuning

---

### Observability

**Observability** answers the question: *"Why is this broken — even if I've never seen this failure before?"*

Observability is a property of a system. A system is **observable** if you can understand its internal state by examining its external outputs (logs, metrics, traces). It enables **exploratory debugging** without deploying new instrumentation.

```
Observability = Ability to ask arbitrary questions about system state at any time
```

**Key insight:** Monitoring tells you *a* problem exists. Observability lets you *understand* the problem.

---

### The Relationship

```
┌─────────────────────────────────────────────────────────────┐
│                      OBSERVABILITY                          │
│                                                             │
│   ┌─────────┐    ┌─────────┐    ┌──────────────────────┐   │
│   │  Logs   │    │ Metrics │    │       Traces         │   │
│   └─────────┘    └─────────┘    └──────────────────────┘   │
│                                                             │
│                  ↓ feeds into ↓                             │
│                                                             │
│   ┌──────────────────────────────────────────────────────┐  │
│   │              MONITORING                              │  │
│   │   Dashboards · Thresholds · Anomaly Detection        │  │
│   └──────────────────────────────────────────────────────┘  │
│                                                             │
│                  ↓ triggers ↓                               │
│                                                             │
│   ┌──────────────────────────────────────────────────────┐  │
│   │              ALERTING                                │  │
│   │   PagerDuty · Slack · On-Call Rotations              │  │
│   └──────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────┘
```

| | Monitoring | Observability |
|---|---|---|
| **Question** | Is X broken? | Why is X broken? |
| **Data model** | Pre-defined metrics | Arbitrary structured events |
| **Failure modes** | Known unknowns | Unknown unknowns |
| **Approach** | Reactive alerting | Exploratory debugging |
| **When useful** | Operational stability | Debugging production incidents |


## 2. The Three Pillars of Observability

<a name="2"></a>

The three pillars provide complementary views into system behaviour. Together they form a complete picture.

```
┌──────────────────────────────────────────────────────────────────────┐
│                    THREE PILLARS OF OBSERVABILITY                    │
│                                                                      │
│  ┌────────────────┐  ┌────────────────┐  ┌────────────────────────┐ │
│  │     METRICS    │  │      LOGS      │  │        TRACES          │ │
│  │                │  │                │  │                        │ │
│  │  Numeric time  │  │ Timestamped    │  │ Request lifecycle      │ │
│  │  series data   │  │ event records  │  │ across services        │ │
│  │                │  │                │  │                        │ │
│  │  "What is the  │  │  "What exactly │  │  "Which service caused │ │
│  │  error rate?"  │  │  happened?"    │  │  this 3s latency?"     │ │
│  └────────────────┘  └────────────────┘  └────────────────────────┘ │
│       Aggregated          Verbose              Contextual            │
│       Cheap to store      Expensive            Moderate cost         │
└──────────────────────────────────────────────────────────────────────┘
```

Each pillar has a different cost/benefit tradeoff:

| Pillar | Storage Cost | Query Speed | Best For |
|--------|-------------|-------------|----------|
| Metrics | Low (aggregated) | Very fast | Dashboards, alerting |
| Logs | High (raw text) | Slow (full-text search) | Debugging, audit trails |
| Traces | Medium (sampled) | Medium | Latency attribution |

The practical workflow:
1. **Metrics alert** fires — "error rate spiked at 14:32"
2. **Logs** show *what* failed — "NullPointerException in OrderService"
3. **Traces** show *where* in the call chain — "DB query in CartService took 4.2s causing cascade"


## 3. Metrics Deep Dive

<a name="3"></a>

Metrics are **numeric measurements over time**. They are the foundation of dashboards and alerting because they are cheap to store and fast to query.

---

### Types of Metrics

#### Counter
- Monotonically increasing value, reset to 0 on restart
- Use for: total requests, total errors, bytes sent
- Query pattern: always use `rate()` or `increase()` — never the raw value

```
http_requests_total{service="api", status="500"} 1042
```

#### Gauge
- A value that can go up or down
- Use for: current connections, memory usage, queue depth, CPU %

```
active_connections{service="db"} 87
memory_usage_bytes{host="prod-1"} 4294967296
```

#### Histogram
- Samples observations and counts them in configurable buckets
- Use for: request duration, response size
- Enables **percentile calculations** (p50, p95, p99)

```
http_request_duration_seconds_bucket{le="0.1"}  2345
http_request_duration_seconds_bucket{le="0.5"}  4567
http_request_duration_seconds_bucket{le="1.0"}  4892
http_request_duration_seconds_bucket{le="+Inf"} 4900
http_request_duration_seconds_sum              987.3
http_request_duration_seconds_count           4900
```

#### Summary
- Similar to histogram but calculates quantiles client-side
- Less flexible for aggregation across instances — prefer histograms

---

### Metric Dimensions (Labels/Tags)

Labels add context to metrics and enable slicing and dicing during incidents.

```
http_requests_total{
  service="checkout",
  endpoint="/api/v1/order",
  method="POST",
  status_code="500",
  region="us-east-1",
  version="v2.3.1"
}
```

**Cardinality warning:** Each unique label combination creates a new time series. High-cardinality labels (user_id, request_id) can cause cardinality explosions that crash your metrics store.

```
# Good labels (bounded cardinality)
service, endpoint, method, status_code, region, environment

# Bad labels (unbounded cardinality — NEVER do this)
user_id, order_id, session_id, ip_address
```

---

### The Four Golden Signals (Google SRE)

Google's Site Reliability Engineering book defines four signals that matter most for any user-facing service:

```
┌────────────────────────────────────────────────────────────────┐
│                   FOUR GOLDEN SIGNALS                          │
│                                                                │
│  1. LATENCY    — How long requests take                        │
│     - Distinguish successful vs. failed latency               │
│     - Use percentiles (p95, p99) not averages                  │
│                                                                │
│  2. TRAFFIC    — Demand on the system                          │
│     - Requests per second, transactions/min                    │
│     - Helps separate load-induced vs. logic errors             │
│                                                                │
│  3. ERRORS     — Rate of failed requests                       │
│     - Explicit (HTTP 5xx) + implicit (wrong data returned)     │
│     - Track as a rate (errors/total), not absolute count       │
│                                                                │
│  4. SATURATION — How "full" the service is                     │
│     - CPU%, memory%, disk I/O, connection pool usage           │
│     - Leading indicator — catches problems before they happen  │
└────────────────────────────────────────────────────────────────┘
```

**Rule of thumb:** If you can only instrument four things per service, instrument these four.

---

### PromQL Examples (Prometheus)

```promql
# Error rate over last 5 minutes
rate(http_requests_total{status=~"5.."}[5m])
  /
rate(http_requests_total[5m])

# 99th percentile latency
histogram_quantile(0.99,
  sum(rate(http_request_duration_seconds_bucket[5m])) by (le, service)
)

# Saturation: connection pool usage %
pg_pool_active_connections / pg_pool_max_connections * 100

# Apdex score (satisfied < 0.3s, tolerated < 1.2s)
(
  sum(rate(http_request_duration_seconds_bucket{le="0.3"}[5m]))
  + sum(rate(http_request_duration_seconds_bucket{le="1.2"}[5m]))
) / 2
/ sum(rate(http_request_duration_seconds_count[5m]))
```


## 4. Logs Deep Dive

<a name="4"></a>

Logs are **timestamped, immutable records of discrete events**. They are the most human-readable of the three pillars and are indispensable for debugging.

---

### Log Levels — What Goes Where

```
TRACE   │ Extremely verbose — individual method calls, variable values
        │ Only enabled during deep debugging. Never in production.
        │
DEBUG   │ Diagnostic info useful during development
        │ Usually disabled in production; toggle dynamically when needed
        │
INFO    │ Normal operational events — service started, job completed,
        │ user logged in. These SHOULD appear in production logs.
        │
WARN    │ Unexpected but recoverable — retry succeeded, using fallback,
        │ near capacity. Investigate when they spike.
        │
ERROR   │ A request/operation failed. Needs attention.
        │ Always include exception + context.
        │
FATAL   │ Service cannot continue — crash imminent or already happened
        │ Rare; usually means "page someone NOW"
```

**Rule of thumb:** In production, log at INFO and above. Enable DEBUG/TRACE dynamically only when investigating a specific incident.

---

### Structured Logging

**Unstructured logs** are hard to query at scale:
```
2024-01-15 14:32:01 ERROR Failed to process order 12345 for user john@example.com: timeout after 5000ms
```

**Structured logs** (JSON) are machine-queryable and correlatable:
```json
{
  "timestamp": "2024-01-15T14:32:01.342Z",
  "level": "ERROR",
  "service": "order-service",
  "version": "v2.3.1",
  "trace_id": "abc123def456",
  "span_id": "789xyz",
  "user_id": "usr_987",
  "order_id": "ord_12345",
  "event": "order_processing_failed",
  "error": "TimeoutError",
  "duration_ms": 5000,
  "message": "Failed to process order: downstream payment service timeout"
}
```

Key fields to **always** include:
- `timestamp` — ISO 8601 with milliseconds
- `level` — severity
- `service` + `version` — which binary produced this log
- `trace_id` + `span_id` — correlation with distributed traces
- `request_id` — correlation within a single service
- `event` — machine-readable event type (snake_case)
- `message` — human-readable description

---

### Log Aggregation Pipeline

```
┌──────────────────────────────────────────────────────────────────────┐
│                     LOG AGGREGATION PIPELINE                         │
│                                                                      │
│  Services         Shipper         Aggregator       Storage/Query     │
│  ─────────        ────────        ──────────        ─────────────    │
│  app-1 ──────────► Fluentd ──────►           ──────► Elasticsearch  │
│  app-2 ──────────► Fluent  ──────► Logstash  ──────► OpenSearch     │
│  app-3 ──────────►  Bit    ──────►           ──────► Loki           │
│  nginx ──────────► Vector  ──────►           ──────► Splunk         │
│  k8s   ──────────►        ──────►           ──────► CloudWatch      │
└──────────────────────────────────────────────────────────────────────┘
```

### Log Retention Strategy

| Log Type | Retention | Rationale |
|----------|-----------|-----------|
| Application errors | 30–90 days | Active debugging window |
| Access logs | 30–90 days | Security investigation |
| Audit logs | 1–7 years | Compliance requirement |
| Debug logs | 3–7 days | Short-lived, high volume |

---

### Correlation ID Pattern

The correlation ID pattern links all log entries for a single user action across all services:

```
Browser → API Gateway → Auth Service → Order Service → Payment Service
            │
            │ Generates: X-Request-ID: req_abc123
            │ (propagated to every downstream call as a header)
            │
All services log: { "request_id": "req_abc123", ... }
```

When an incident occurs, search `request_id: "req_abc123"` to see every log line from every service that participated in that one user request — even across service boundaries.


## 5. Distributed Tracing Deep Dive

<a name="5"></a>

In a microservices architecture, a single user request fans out across dozens of services. **Distributed tracing** tracks that request end-to-end, showing exactly where time was spent.

---

### Core Concepts

```
Trace = the entire lifecycle of one request (has a unique trace_id)
  │
  ├── Span = one unit of work within a trace (has span_id + parent_span_id)
  │     ├── Duration
  │     ├── Tags/Attributes (key-value metadata)
  │     └── Events/Logs (timestamped annotations within the span)
  │
  └── Context Propagation = mechanism to pass trace_id across service boundaries
        (HTTP headers, message queue metadata, gRPC metadata)
```

### A Trace Visualised as a Gantt Chart

```
Trace: req_abc123 — Total: 452ms
│
├── [0ms  → 452ms] API Gateway                             (452ms)
│     ├── [10ms → 60ms]  Auth Service                      (50ms)
│     │     └── [15ms → 55ms]  Redis: token_lookup          (40ms)
│     │
│     └── [65ms → 450ms] Order Service                     (385ms)
│           ├── [70ms → 120ms]  Inventory Service           (50ms)
│           │     └── [72ms → 118ms]  Postgres: SELECT      (46ms)
│           │
│           ├── [125ms → 430ms] Payment Service             (305ms)  ← SLOW
│           │     ├── [130ms → 135ms] Fraud Check           (5ms)
│           │     └── [140ms → 428ms] Stripe API call       (288ms)  ← ROOT CAUSE
│           │
│           └── [431ms → 448ms] Notification Service        (17ms)
```

Without tracing, you'd see a 452ms request — with tracing you instantly know: **Stripe API is the bottleneck, taking 288ms out of 452ms total.**

---

### Context Propagation Standards

The industry has converged on **W3C TraceContext** as the standard format:

```
traceparent: 00-4bf92f3577b34da6a3ce929d0e0e4736-00f067aa0ba902b7-01
              ^  ^                                ^               ^
              |  trace-id (128-bit hex)           span-id         sampled flag
              version
```

OpenTelemetry is the de-facto standard SDK for generating and propagating traces.

---

### Sampling Strategies

Tracing every request is expensive. Sampling controls which requests get traced:

| Strategy | How It Works | Trade-off |
|----------|-------------|-----------|
| **Head sampling** | Decision made at trace start (e.g., 1%) | Simple; may miss rare errors |
| **Tail sampling** | Decision made after trace completes | Captures errors always; more complex |
| **Adaptive sampling** | Rate adjusts based on traffic volume | Best of both; needs a collector |

**Recommended:** Use tail-based sampling — always sample traces that contain errors or exceed a latency threshold.

```python
# OpenTelemetry Python example
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

provider = TracerProvider()
provider.add_span_processor(
    BatchSpanProcessor(OTLPSpanExporter(endpoint="http://otel-collector:4317"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer(__name__)

def process_order(order_id: str):
    with tracer.start_as_current_span("process_order") as span:
        span.set_attribute("order.id", order_id)
        span.set_attribute("order.service", "checkout")
        try:
            result = charge_payment(order_id)
            span.set_attribute("payment.status", "success")
            return result
        except PaymentError as e:
            span.record_exception(e)
            span.set_status(trace.StatusCode.ERROR, str(e))
            raise
```


## 6. What to Measure — Golden Signals, USE & RED Methods

<a name="6"></a>

Three complementary frameworks help you decide *what* to instrument:

---

### Google's Four Golden Signals (User-Facing Services)

Already covered in Section 3. Summary:

| Signal | What it measures | Example metric |
|--------|-----------------|----------------|
| Latency | How long requests take | `p99(request_duration_seconds)` |
| Traffic | How much demand exists | `rate(requests_total[5m])` |
| Errors | How often requests fail | `rate(errors_total[5m]) / rate(requests_total[5m])` |
| Saturation | How close to capacity | `cpu_usage_percent`, `queue_depth` |

---

### USE Method (Brendan Gregg — Infrastructure/Resource Monitoring)

For every **resource** (CPU, memory, disk, network, database connections):

- **Utilization** — % time the resource is busy (`cpu_usage = 80%`)
- **Saturation** — queue depth or backlog (`run_queue_length = 12`)
- **Errors** — count of error events (`disk_io_errors_total`)

```
USE = Utilization + Saturation + Errors

Best for: Hardware resources, OS-level metrics, database connection pools
```

Practical application — for each infrastructure component ask:
```
┌─────────────┬──────────────────────┬────────────────────┬───────────────────┐
│  Resource   │     Utilization      │    Saturation      │      Errors       │
├─────────────┼──────────────────────┼────────────────────┼───────────────────┤
│ CPU         │ % busy               │ run queue length   │ N/A               │
│ Memory      │ % used               │ swap usage         │ OOM kills         │
│ Disk        │ % I/O time           │ I/O queue depth    │ I/O errors        │
│ Network NIC │ % bandwidth          │ TX/RX drops        │ TX/RX errors      │
│ DB Connections│ connections used   │ wait queue size    │ connection errors  │
└─────────────┴──────────────────────┴────────────────────┴───────────────────┘
```

---

### RED Method (Tom Wilkie — Microservice Monitoring)

For every **microservice** or **API endpoint**:

- **Rate** — requests per second (how many requests are being handled?)
- **Errors** — error rate (what fraction of requests are failing?)
- **Duration** — distribution of request durations (how fast are we?)

```
RED = Rate + Errors + Duration

Best for: Microservices, API endpoints, user-facing services
```

RED is essentially a focused version of the Golden Signals, optimised for service-level metrics.

---

### Choosing the Right Method

```
                   Are you measuring...
                          │
           ┌──────────────┴──────────────┐
           │                             │
    Infrastructure                  Microservice
    (CPU, disk, DB)               (API, function)
           │                             │
        USE Method                   RED Method
     Util+Sat+Errors            Rate+Errors+Duration
```

In practice, you use both: USE for your infrastructure layer, RED for your application layer, and Golden Signals to communicate with stakeholders.

---

### Apdex Score — User Satisfaction Index

Apdex (Application Performance Index) converts raw latency into a satisfaction score from 0 to 1:

```
Define T = target response time (e.g., 200ms)

Satisfied  = requests < T        → full score (1.0)
Tolerating = requests T to 4T    → half score (0.5)
Frustrated = requests > 4T       → no score (0.0)

Apdex = (Satisfied + 0.5 × Tolerating) / Total
```

| Score | Interpretation |
|-------|---------------|
| 1.0 | Excellent — all users satisfied |
| 0.85–0.99 | Good |
| 0.70–0.84 | Fair — investigate |
| < 0.70 | Poor — users are unhappy, act now |


## 7. SLIs, SLOs, SLAs — The Reliability Contract

<a name="7"></a>

This trio defines the formal reliability contract between engineers and the business. Getting this right is what separates reactive fire-fighting from proactive reliability engineering.

---

### Definitions

#### SLI — Service Level Indicator
A **quantitative measure** of some aspect of the service's behaviour.
The *raw metric* you actually measure.

```
SLI = (Good Events) / (Total Events)

Examples:
  - % of HTTP requests completing in < 200ms
  - % of requests returning a non-5xx response
  - % of background jobs completing successfully within 1 hour
```

#### SLO — Service Level Objective
A **target value** or range for an SLI, over a rolling time window.
The *promise you make to yourself* (internal commitment).

```
SLO = SLI target over a time window

Examples:
  - 99.9% of requests complete in < 200ms, measured over a 30-day rolling window
  - 99.5% availability, measured weekly
  - 99.0% of jobs complete within 1 hour
```

#### SLA — Service Level Agreement
A **contractual commitment** to customers with defined consequences for violation (refunds, credits, penalties).
The *promise you make to customers* (external, legal).

```
SLA = SLO with consequences

Example:
  - 99.9% monthly uptime or customer receives service credits
```

---

### The Relationship

```
SLI (measurement) → SLO (internal target) → SLA (external contract)

         SLA is always looser than SLO
         SLO is always looser than what you can actually achieve
         
         If SLO = 99.9%, a safe SLA = 99.5%
         This buffer lets you investigate + fix before breaching the SLA
```

---

### Error Budget

The **error budget** is the single most important concept in SRE. It makes reliability a shared engineering resource rather than a "devs vs. ops" argument.

```
Error Budget = 1 - SLO

If SLO = 99.9%  →  Error Budget = 0.1%
Over 30 days = 43,200 minutes × 0.001 = 43.2 minutes of allowable downtime
```

**How error budgets change team dynamics:**

```
Error budget HEALTHY (lots remaining):
  → Ship new features quickly
  → Try risky experiments
  → Fewer restrictions on deployments

Error budget DEPLETED (near or at 0):
  → Freeze new feature releases
  → Focus engineering effort on reliability work
  → Conduct reliability sprint to recover budget
```

This turns reliability into a conversation the product team and engineering team have together — it's not just "ops' problem."

---

### Practical SLO Design

**Step 1: Choose the right SLI type**

| User Action | SLI to track |
|-------------|-------------|
| HTTP request | Availability (success rate) + Latency |
| Data pipeline | Freshness (data age) + Completeness (% rows processed) |
| Storage | Durability (reads return what was written) |
| Background job | Throughput (jobs completed per hour) |

**Step 2: Set a realistic target**
- Start with what you currently achieve (check last 30 days)
- Set the SLO *slightly* below current performance
- Tighten it over quarters as you improve reliability

**Step 3: Define the compliance window**
- Rolling 28/30 days is most common (avoids calendar effects)
- Shorter windows (7 days) give faster feedback but more noise

**Step 4: Build error budget burn rate alerts** (see Section 8)

---

### Example SLO Document

```yaml
# SLO: Checkout API
service: checkout-api
owner: platform-team

slo:
  - name: availability
    description: "% of requests returning non-5xx"
    sli: |
      sum(rate(http_requests_total{service="checkout",status!~"5.."}[5m]))
      /
      sum(rate(http_requests_total{service="checkout"}[5m]))
    target: 0.999       # 99.9%
    window: 30d

  - name: latency_p99
    description: "99th percentile latency < 500ms"
    sli: |
      histogram_quantile(0.99,
        rate(http_request_duration_seconds_bucket{service="checkout"}[5m])
      ) < 0.5
    target: 0.995       # 99.5% of the time p99 < 500ms
    window: 30d

error_budget_policy:
  - burn_rate_threshold: 14.4x   # Budget gone in 2 hours → page immediately
    action: page on-call
  - burn_rate_threshold: 6x      # Budget gone in 5 hours → ticket + Slack alert
    action: create incident ticket
  - burn_rate_threshold: 1x      # Budget gone in 30 days → weekly review
    action: add to reliability backlog
```


## 8. Alerting — Turning Signals into Action

<a name="8"></a>

Alerting is where monitoring meets human response. Bad alerting is worse than no alerting — it trains engineers to ignore pages.

---

### The Alert Quality Bar

Every alert should pass this test before you create it:

```
Is this alert ACTIONABLE?
  → Can a human do something right now to fix or mitigate this?
  → If no, it's informational — put it in a dashboard, not an alert

Is this alert URGENT?
  → Does a human need to know NOW (i.e., cannot wait until morning)?
  → If no, it's a ticket/email, not a page

Is this alert MEANINGFUL?
  → Does it correlate with real user impact?
  → If no, you're alerting on symptoms, not causes
```

**The two failure modes of alerting:**

| Problem | Symptom | Root Cause |
|---------|---------|------------|
| Alert fatigue | Engineers ignore pages | Too many alerts, especially noise |
| Silent failures | Outages not caught | Monitoring gaps |

Alert fatigue is the more common and dangerous problem. An on-call engineer who sees 50 pages per shift will start dismissing them — including the critical one.

---

### Symptom-Based vs Cause-Based Alerting

**Cause-based alerts** (bad):
```
ALERT: CPU > 80%
ALERT: Memory > 90%
ALERT: Disk I/O > 100MB/s
```
These require deep knowledge to interpret. CPU at 80% could be completely fine (batch job running) or catastrophic (infinite loop). These belong in dashboards.

**Symptom-based alerts** (good):
```
ALERT: Error rate > 1% for 5 consecutive minutes
ALERT: p99 latency > 2 seconds for 3 minutes
ALERT: Less than 100 orders processed in the last 10 minutes
```
These directly describe user impact. The cause can be anything — but the *symptom* (users experiencing errors) is unambiguously bad.

**Rule:** Alert on symptoms (user impact). Use dashboards + logs + traces to find causes.

---

### Alert Severity Levels

```
┌──────────────┬─────────────────────────────────────────────────────────┐
│  SEVERITY    │  DEFINITION & RESPONSE                                  │
├──────────────┼─────────────────────────────────────────────────────────┤
│  P0 CRITICAL │ Total service outage. All users affected.               │
│              │ Response: Wake up on-call NOW. All hands.               │
│              │ SLA violation imminent or occurring.                    │
├──────────────┼─────────────────────────────────────────────────────────┤
│  P1 HIGH     │ Significant degradation. Many users affected.           │
│              │ Response: Page on-call. Fix within 30 min.              │
│              │ Error budget burning fast.                              │
├──────────────┼─────────────────────────────────────────────────────────┤
│  P2 MEDIUM   │ Partial degradation. Some users affected.               │
│              │ Response: Slack notification. Fix within 4 hours.       │
│              │ Error budget burning at elevated rate.                  │
├──────────────┼─────────────────────────────────────────────────────────┤
│  P3 LOW      │ Minor issue. No current user impact.                    │
│              │ Response: Create ticket. Fix in next sprint.            │
│              │ Error budget burning at normal rate.                    │
└──────────────┴─────────────────────────────────────────────────────────┘
```

---

### Error Budget Burn Rate Alerting (Google SRE)

This is the most powerful alerting strategy. Instead of alerting on raw thresholds, alert on how fast the error budget is burning.

```
Burn rate = current error rate / (1 - SLO)

If SLO = 99.9% → error budget = 0.1%

Burn rate = 1x  → budget depleted in exactly 30 days (normal)
Burn rate = 14.4x → budget depleted in 2 hours  → PAGE
Burn rate = 6x    → budget depleted in 5 hours  → PAGE (if sustained > 1h)
Burn rate = 3x    → budget depleted in 10 days  → ticket
```

**Two-alert strategy (short + long window):**

The short window catches fast-burning incidents. The long window catches slow burns that accumulate unnoticed.

```yaml
# Prometheus alerting rules

# P1 — fast burn (budget gone in 2h)
- alert: HighErrorBudgetBurn
  expr: |
    (
      job:slo_errors:rate5m{job="checkout"} / 0.001 > 14.4
    AND
      job:slo_errors:rate1h{job="checkout"} / 0.001 > 14.4
    )
  for: 2m
  labels:
    severity: page
  annotations:
    summary: "High error budget burn rate: checkout API"
    description: "Error budget will be exhausted in ~2 hours at current rate"

# P2 — slow burn (budget gone in 5 days)
- alert: MediumErrorBudgetBurn
  expr: |
    (
      job:slo_errors:rate30m{job="checkout"} / 0.001 > 3
    AND
      job:slo_errors:rate6h{job="checkout"} / 0.001 > 3
    )
  for: 15m
  labels:
    severity: ticket
  annotations:
    summary: "Elevated error budget burn: checkout API"
```

---

### Alert Routing & On-Call

```
┌────────────────────────────────────────────────────────────────┐
│                   ALERTING PIPELINE                            │
│                                                                │
│  Prometheus → Alertmanager → Route by severity/team           │
│                    │                                           │
│                    ├── P0/P1 → PagerDuty → Wake up on-call    │
│                    │          (escalation policy: 5min ack)   │
│                    │                                           │
│                    ├── P2    → Slack #incidents channel        │
│                    │          + create Jira ticket             │
│                    │                                           │
│                    └── P3    → Email digest / Jira backlog     │
└────────────────────────────────────────────────────────────────┘
```

**On-call best practices:**
- **Rotation:** Spread on-call across the team (weekly rotations)
- **Shadowing:** New engineers shadow before going solo
- **Runbooks:** Every P0/P1 alert links to a runbook with step-by-step triage
- **Post-incident review:** Every P0 triggers a blameless post-mortem
- **Compensation:** On-call outside business hours should be compensated

---

### Runbook Template

Every actionable alert should link to a runbook:

```markdown
# Runbook: High Error Rate — Checkout API

## Alert
- Alert name: HighErrorBudgetBurn
- Severity: P1
- Channel: #incidents

## Symptoms
- Error rate > 1% on checkout-api
- Users reporting payment failures

## Immediate Triage (< 5 minutes)
1. Check Grafana: [Checkout Dashboard](link)
2. Check recent deployments: `kubectl rollout history deployment/checkout`
3. Check DB connections: `psql -c "SELECT count(*) FROM pg_stat_activity"`

## Mitigation Options
- **If recent deploy:** Roll back with `kubectl rollout undo deployment/checkout`
- **If DB issue:** Increase connection pool or failover to read replica
- **If downstream:** Enable circuit breaker for payment service

## Escalation
- Not resolved in 15min → escalate to #checkout-team + tag @team-lead
- Suspected data corruption → escalate to #data-team immediately

## Post-Incident
- File incident report in [Incident Tracker](link)
- Schedule post-mortem within 48 hours
```


## 9. Dashboards & Visualization

<a name="9"></a>

A good dashboard answers a specific question in under 30 seconds. A bad dashboard is a sea of numbers nobody looks at until something is already on fire.

---

### Dashboard Hierarchy

Design dashboards in layers. Each layer drills deeper:

```
┌─────────────────────────────────────────────────────────────────────┐
│  LAYER 1: Executive / Service Health                                │
│  "Is everything OK right now?"                                      │
│  Audience: Managers, SREs on rotation, NOC                         │
│  Metrics: SLO burn rate, overall availability, P0 incidents         │
│                         ↓ drill down                                │
├─────────────────────────────────────────────────────────────────────┤
│  LAYER 2: Service Overview (one per service)                        │
│  "How is this service performing?"                                  │
│  Audience: Service owners, on-call engineers                        │
│  Metrics: Rate, errors, latency (RED), saturation                  │
│                         ↓ drill down                                │
├─────────────────────────────────────────────────────────────────────┤
│  LAYER 3: Deep Dive / Component                                     │
│  "What's causing this problem?"                                     │
│  Audience: Engineers actively debugging                             │
│  Metrics: Per-endpoint breakdowns, DB query times, cache hit rates  │
└─────────────────────────────────────────────────────────────────────┘
```

---

### Dashboard Design Principles

**1. Lead with the most important signal**
Put error rate and latency at the top-left. Humans read top-left first.

**2. Group related metrics together**
Request metrics (rate/errors/latency) in one row. Infrastructure (CPU/memory) in another.

**3. Use consistent time ranges**
Lock dashboards to a default window (last 6h for operations, last 30d for SLO tracking).

**4. Annotate deployments**
Add vertical lines at deployment times. This instantly correlates "when did this spike start" with "what changed."

**5. Use heatmaps for latency, not averages**
Averages hide bimodal distributions. A heatmap (or at minimum p50/p95/p99) shows the full picture.

**6. Include relevant links**
Every dashboard panel should link to:
- The relevant runbook
- The relevant log search (pre-built query)
- The relevant traces

---

### Golden Dashboard Layout (Grafana)

```
┌──────────────────────────────────────────────────────────────────────┐
│  Service: checkout-api  │  Environment: prod  │  Last 6h            │
├──────────────────────────────────────────────────────────────────────┤
│  ┌─────────────┐  ┌─────────────┐  ┌──────────────┐  ┌──────────┐  │
│  │ Error Rate  │  │  p99 Latency│  │  Req/sec     │  │ SLO Burn │  │
│  │   0.03%     │  │   187ms     │  │   1,240 rps  │  │  0.8x    │  │
│  │  ▁▁▂▁▁▁▁▁  │  │  ▂▂▂▃▂▂▂▂  │  │  ▄▄▄▄▄▄▄▄   │  │ HEALTHY  │  │
│  └─────────────┘  └─────────────┘  └──────────────┘  └──────────┘  │
├──────────────────────────────────────────────────────────────────────┤
│  Request Rate by Endpoint      │  Latency Heatmap                   │
│  ┌──────────────────────────┐  │  ┌─────────────────────────────┐   │
│  │ /api/order     ████████  │  │  │ fast  ■■■■■■■■■□□□□□□□□□□  │  │
│  │ /api/checkout  ██████    │  │  │ med   ■■■■■■□□□□□□□□□□□□□□  │  │
│  │ /api/cart      ███       │  │  │ slow  ■□□□□□□□□□□□□□□□□□□□  │  │
│  └──────────────────────────┘  │  └─────────────────────────────┘   │
├──────────────────────────────────────────────────────────────────────┤
│  Infrastructure                                                      │
│  CPU 43%  ▂▂▂▃▂▂▂▂    Memory 68%  ▅▅▅▅▅▅▅▅    DB Pool 31/100       │
└──────────────────────────────────────────────────────────────────────┘
```

---

### What NOT to put in a dashboard

- Raw counter values (always use rates)
- Metrics nobody has acted on in the last 3 months (delete them)
- Duplicate panels showing the same signal in different ways
- Averages of latency (always use percentiles)
- Metrics without units or descriptions


## 10. Incident Detection Workflow

<a name="10"></a>

From signal to resolution — the complete lifecycle of an incident.

---

### Incident Lifecycle

```
┌──────────────────────────────────────────────────────────────────────────┐
│                        INCIDENT LIFECYCLE                                │
│                                                                          │
│  1. DETECT          2. TRIAGE          3. MITIGATE        4. RESOLVE    │
│  ──────────         ──────────         ────────────        ──────────   │
│  Alert fires        Is this real?      Stop the            Fix the      │
│  or user report     How bad is it?     bleeding            root cause   │
│                     Who is affected?                                     │
│                          │                                               │
│                    5. COMMUNICATE     6. POST-MORTEM                    │
│                    ──────────────     ──────────────                    │
│                    Status page        Blameless review                  │
│                    Stakeholder        5 Whys analysis                   │
│                    updates            Action items                      │
└──────────────────────────────────────────────────────────────────────────┘
```

---

### Detection Methods

Not all incidents are detected by automated monitoring:

| Detection Method | % of Real Incidents | Notes |
|-----------------|---------------------|-------|
| Automated alert (metrics) | ~60% | Most common for infrastructure issues |
| User/customer report | ~30% | Catches silent failures monitoring misses |
| Automated alert (logs) | ~5% | Error log rate spikes |
| External monitoring (synthetic) | ~5% | Uptime checks from outside |

**Synthetic monitoring (outside-in):** Run scripted user journeys from external locations every 1-5 minutes. These catch issues automated internal monitoring misses (CDN problems, DNS failures, region-specific issues).

```python
# Synthetic check example (Playwright)
async def synthetic_checkout_flow():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        
        start = time.time()
        await page.goto("https://myapp.com")
        await page.click("text=Add to Cart")
        await page.click("text=Checkout")
        await page.fill("#card-number", "4111111111111111")
        await page.click("text=Place Order")
        await page.wait_for_selector("text=Order Confirmed")
        duration = time.time() - start
        
        # Emit as metric
        record_gauge("synthetic.checkout.duration_seconds", duration)
        record_counter("synthetic.checkout.success_total", 1)
```

---

### Incident Triage Checklist

When an alert fires, run through this in order:

```
[ ] 1. Is the alert real or spurious?
        - Check if multiple data points confirm the issue
        - Check if related services are also affected
        - Verify it's not a known deployment or maintenance

[ ] 2. What is the blast radius?
        - How many users are affected? (% of traffic)
        - Which features/endpoints are broken?
        - Is this complete outage or partial degradation?

[ ] 3. When did it start?
        - Check metric graphs for exact onset time
        - Look for deployment annotations near onset time
        - Check recent config changes

[ ] 4. What changed?
        - Recent deployments (git log, deployment annotations)
        - Recent config/infrastructure changes
        - Upstream service changes (dependency dashboards)
        - Traffic pattern changes (DDoS? marketing campaign?)

[ ] 5. Can it be mitigated immediately?
        - Roll back the last deployment?
        - Disable the affected feature via feature flag?
        - Scale up the affected service?
        - Failover to secondary region/replica?
```

---

### MTTD and MTTR — The Two Key Reliability Metrics

**MTTD — Mean Time to Detect**
How long from an incident starting until it's detected.

```
MTTD = avg(detection_time - incident_start_time)

Target: < 5 minutes for P0, < 30 minutes for P1
Ways to improve: Better alerting coverage, synthetic monitoring, lower alert thresholds
```

**MTTR — Mean Time to Resolve**
How long from detection until service is restored.

```
MTTR = avg(resolution_time - detection_time)

Target: < 30 minutes for P0, < 4 hours for P1
Ways to improve: Better runbooks, automated rollback, on-call training
```

Note: MTTR measures *mitigation* (service restored), not *root cause fix* (which takes much longer).

---

### Blameless Post-Mortem Template

```markdown
# Post-Mortem: [Incident Title]

**Date:** YYYY-MM-DD
**Duration:** HH:MM to HH:MM (X hours Y minutes)
**Severity:** P0 / P1
**Impact:** % users affected, revenue impact, SLO budget consumed

## Timeline
| Time | Event |
|------|-------|
| 14:32 | First alert fired (HighErrorBudgetBurn) |
| 14:34 | On-call acknowledged |
| 14:40 | Root cause identified: bad deploy |
| 14:45 | Rollback initiated |
| 14:52 | Service restored |
| 15:30 | Root cause confirmed via logs |

## Root Cause
[One paragraph: what actually went wrong]

## Contributing Factors
- [Factor 1: e.g., no automated canary analysis before full rollout]
- [Factor 2: e.g., test environment didn't cover this code path]

## 5 Whys Analysis
- Why did users see errors? → Payment service returned 500s
- Why did payment service return 500s? → New code threw uncaught exception
- Why was there an uncaught exception? → Edge case not handled
- Why wasn't the edge case tested? → No test for null payment_method
- Why was there no test? → Test coverage for this module was below standard

## Action Items
| Item | Owner | Due Date | Priority |
|------|-------|----------|----------|
| Add integration test for null payment_method | @alice | 2024-02-01 | P0 |
| Add canary analysis step to deploy pipeline | @bob | 2024-02-15 | P1 |
| Lower alert threshold for payment errors | @alice | 2024-01-25 | P1 |

## What Went Well
- On-call detected and responded within 2 minutes of alert
- Rollback procedure worked smoothly

## What Went Poorly
- MTTD was 8 minutes — alert threshold was too high
- No runbook existed for this failure mode
```


## 11. Tooling Landscape

<a name="11"></a>

The observability ecosystem is large. Here's how the major tools fit together:

---

### Full Observability Stack Options

#### Option A: Open Source (Self-Hosted)
```
Metrics:  Prometheus + Grafana
Logs:     Loki + Grafana  (or ELK: Elasticsearch + Logstash + Kibana)
Traces:   Jaeger or Tempo + Grafana
Alerts:   Prometheus Alertmanager → PagerDuty/OpsGenie
Instrumentation: OpenTelemetry (vendor-neutral SDKs)

Cost: Low (infra only). Maintenance: High.
```

#### Option B: Cloud-Native (Managed)
```
AWS:    CloudWatch (metrics+logs) + X-Ray (traces)
GCP:    Cloud Monitoring + Cloud Logging + Cloud Trace
Azure:  Azure Monitor + Application Insights

Cost: Pay-per-use, can get expensive at scale.
Maintenance: Low (managed service).
```

#### Option C: Commercial (All-in-One)
```
Datadog     — Full stack: metrics, logs, traces, APM, dashboards
New Relic   — Full stack with ML-based anomaly detection
Dynatrace   — AI-driven automatic instrumentation
Honeycomb   — Optimised for high-cardinality observability
Splunk      — Enterprise logs + SIEM

Cost: High. Maintenance: Low. Best integration.
```

---

### Tool Comparison

| Tool | Category | Strengths | Weaknesses |
|------|----------|-----------|------------|
| **Prometheus** | Metrics | Pull-based, powerful PromQL, huge ecosystem | No long-term storage by default |
| **Grafana** | Visualization | Connects to everything, flexible | No built-in data storage |
| **Loki** | Logs | Cheap (label-indexed, not full-text) | Limited query capability vs ES |
| **Elasticsearch** | Logs | Powerful full-text search | Expensive, operationally complex |
| **Jaeger** | Traces | Open source, battle-tested | UI not as polished as commercial |
| **Tempo** | Traces | Grafana-native, cheap storage | Less mature than Jaeger |
| **OpenTelemetry** | Instrumentation | Vendor-neutral, industry standard | Complex to set up |
| **Datadog** | All-in-one | Best UX, correlates signals | Very expensive at scale |
| **PagerDuty** | Alerting/On-call | Best-in-class on-call management | Expensive |
| **OpsGenie** | Alerting/On-call | Good Atlassian integration | UX not as good as PagerDuty |

---

### OpenTelemetry — The Instrumentation Standard

OpenTelemetry (OTel) is the CNCF standard for emitting metrics, logs, and traces. Use it to avoid vendor lock-in.

```
┌───────────────────────────────────────────────────────────────────────┐
│                      OPENTELEMETRY ARCHITECTURE                       │
│                                                                       │
│  Application                                                          │
│  ┌─────────────────────────────────┐                                  │
│  │  OTel SDK (auto + manual instr.)│                                  │
│  │  - Traces (spans)               │                                  │
│  │  - Metrics (counters, gauges)   │   OTLP Protocol                  │
│  │  - Logs (structured events)     │──────────────────►               │
│  └─────────────────────────────────┘                                  │
│                                                                       │
│  OTel Collector (optional but recommended)                            │
│  ┌─────────────────────────────────┐                                  │
│  │  Receive → Process → Export     │──────► Jaeger / Tempo (traces)  │
│  │  - Batching                     │──────► Prometheus (metrics)     │
│  │  - Sampling                     │──────► Loki / ES (logs)         │
│  │  - PII scrubbing                │──────► Datadog / New Relic      │
│  └─────────────────────────────────┘                                  │
└───────────────────────────────────────────────────────────────────────┘
```

The collector as a sidecar/deployment means you can swap backend tools without touching application code.

---

### Recommended Stack for a New Project

```
Stage 1 (MVP / early startup):
  Metrics + Logs + Alerts: Datadog or New Relic (operational simplicity)
  On-call: PagerDuty (free tier or basic)
  
Stage 2 (growing, cost-conscious):
  Metrics: Prometheus + Grafana
  Logs: Grafana Loki
  Traces: Grafana Tempo
  Instrumentation: OpenTelemetry
  On-call: OpsGenie or PagerDuty
  
Stage 3 (large scale, platform team):
  Same as Stage 2 but with:
  - Thanos or Cortex for long-term Prometheus storage
  - Dedicated observability platform team
  - Custom SLO dashboards
  - Automated anomaly detection
```


## 12. Common Pitfalls

<a name="12"></a>

Things that look reasonable but consistently cause problems in practice.

---

### Pitfall 1: Alerting on Averages

```
BAD:  Alert when average latency > 500ms
GOOD: Alert when p99 latency > 500ms
```

Average latency of 100ms sounds great — but if 1% of users are getting 10 second responses, that's thousands of frustrated users. Averages mask the long tail that real users experience.

---

### Pitfall 2: Metric Cardinality Explosions

Adding high-cardinality labels to metrics causes the metric store to run out of memory and crash.

```
BAD:  http_requests_total{user_id="usr_12345", ...}
      → 1 million users = 1 million time series per endpoint

GOOD: http_requests_total{service="api", endpoint="/checkout", status="200"}
      → bounded set of values per label
```

If you need per-user debugging, use logs (filter by user_id in a log query), not metrics.

---

### Pitfall 3: Alert on Cause, Not Symptom

```
BAD:  "Alert when CPU > 80%"
      → CPU at 80% during normal batch job = noisy false positive

GOOD: "Alert when error rate > 0.5% for 5 minutes"
      → Only fires when users are actually affected
```

CPU at 80% is a *potential cause* of user impact. Alert on the impact (errors, latency) and use dashboards to investigate causes.

---

### Pitfall 4: Missing the "for" Duration

Without a minimum duration on alerts, brief spikes cause false positives:

```yaml
# Bad: fires on any single-sample spike
- alert: HighErrorRate
  expr: rate(http_errors_total[5m]) / rate(http_requests_total[5m]) > 0.01

# Good: must be sustained for 5 minutes
- alert: HighErrorRate
  expr: rate(http_errors_total[5m]) / rate(http_requests_total[5m]) > 0.01
  for: 5m   # ← this is critical
```

---

### Pitfall 5: Logging PII

Structured logs often contain more user data than developers realise. This creates compliance (GDPR/CCPA) and security risks.

```
BAD:  { "user": "john@example.com", "card": "4111111111111111", ... }
GOOD: { "user_id": "usr_987", "payment_method_type": "card", ... }
```

Log identifiers (user_id, order_id), not PII (email, name, card numbers). Add PII scrubbing in the log aggregation pipeline as a defence-in-depth measure.

---

### Pitfall 6: No Alert for "Nothing is Happening"

Monitoring systems alert on too-much activity but miss complete silence:

```
# Service processes 1000 jobs/hour normally
# If service crashes silently, it processes 0 jobs — no errors, no alerts

GOOD: Alert when jobs_processed_total does not increase for 10 minutes
      expr: rate(jobs_processed_total[10m]) == 0
```

Watch for the absence of expected events, not just the presence of error events.

---

### Pitfall 7: Chasing MTTD with Too-Low Thresholds

Lowering alert thresholds to reduce detection time causes alert fatigue if thresholds are too aggressive:

```
Bad loop:
1. Set threshold low → too many false positives
2. Engineers start ignoring alerts → miss real incidents
3. Set threshold high → miss real incidents
4. Executives demand better detection → go to step 1

Break the loop with:
- Error budget burn rate alerts (dynamic thresholds)
- Separate fast-burn (page) from slow-burn (ticket)
- Regular alert review meetings ("alert audits")
```

---

### Quick Reference Card

```
┌─────────────────────────────────────────────────────────────────────┐
│                    OBSERVABILITY QUICK REFERENCE                    │
│                                                                     │
│  WHAT TO MEASURE     WHAT TO ALERT ON          WHAT TO LOG         │
│  ─────────────────   ────────────────────────   ──────────────────  │
│  Errors (rate)       Error budget burn rate     Structured JSON     │
│  Latency (p99)       Symptom-based signals      Include trace_id   │
│  Traffic (rps)       Minimum 5min "for:"        No PII in logs     │
│  Saturation (%)      P0/P1 get paged            request_id always  │
│                      P2/P3 get ticketed                            │
│                                                                     │
│  METRIC TYPES        SLO FORMULA                                   │
│  ─────────────────   ──────────────────────────                    │
│  Counter (rate())    Error Budget = 1 - SLO                        │
│  Gauge (current)     Burn Rate = current / (1-SLO)                 │
│  Histogram (p99)     Page if burn rate > 14.4x                     │
│                      Ticket if burn rate > 3x                      │
│                                                                     │
│  THREE PILLARS       FOUR GOLDEN SIGNALS        USE METHOD         │
│  ─────────────────   ────────────────────────   ──────────────────  │
│  Metrics (alerts)    Latency                    Utilization        │
│  Logs (debug)        Traffic                    Saturation         │
│  Traces (perf)       Errors                     Errors             │
│                      Saturation                                     │
└─────────────────────────────────────────────────────────────────────┘
```
